In [4]:
# import required libraries. fluxnet_shuttle needs to be installed

import os
import pandas as pd
from fluxnet_shuttle import listall, download

snapshot_dir = "/data/FLUXNET/metadata"
download_dir = "/data/FLUXNET/raw_zips"
os.makedirs(snapshot_dir, exist_ok=True)
os.makedirs(download_dir, exist_ok=True)

In [5]:
# Discover all available data by querying the ICOS hub

csv_filename = await listall(data_hubs=["icos"], output_dir=snapshot_dir)
catalog = pd.read_csv(csv_filename)
print(f"Found {len(catalog)} sites")
catalog.head()

Found 342 sites


,data_hub,site_id,site_name,location_lat,location_long,igbp,network,team_member_name,team_member_role,team_member_email,first_year,last_year,download_link,fluxnet_product_name,product_citation,product_id,oneflux_code_version,product_source_network
0,ICOS,ZA-Uby,Umhlabuyalingana,-27.395200,32.58890,GRA,EFTEON SAEON;European Fluxes Database,Gregor Feig;Kathleen Smart;Gregor Feig;Kathlee...,PI;PI;PI;PI,gt.feig@saeon.nrf.ac.za;kg.smart@saeon.nrf.ac....,2023,2024,https://data.icos-cp.eu/licence_accept?ids=%5B...,SAEON_ZA-Uby_FLUXNET_2023-2024_v1.3_r1.zip,(2025). Fluxnet Archive Product from Umhlabuya...,STLJfs_nkGgsI6sPzHNIgYE0,v1.3,SAEON
1,ICOS,ZA-Spk,Spioenkop,-28.704080,29.52614,GRA,EFTEON SAEON;European Fluxes Database,Gregor Feig;Abri de Buys;Kathleen Smart;Jeremy...,PI;Technician;PI;Technician;PI;Technician;PI;T...,gt.feig@saeon.nrf.ac.za;aj.debuys@saeon.nrf.ac...,2023,2024,https://data.icos-cp.eu/licence_accept?ids=%5B...,SAEON_ZA-Spk_FLUXNET_2023-2024_v1.3_r1.zip,(2025). Fluxnet Archive Product from Spioenkop...,9DBU3ZCobRStxNAcLBAZVtok,v1.3,SAEON
2,ICOS,ZA-Jks,Jonkershoek,-33.990290,18.95543,GRA,EFTEON SAEON;European Fluxes Database,Kathleen Smart;Warren Joubert;Sagwati Maswanga...,PI;PI;Technician;Technician;PI;PI;Technician;T...,kg.smart@saeon.nrf.ac.za;wr.joubert@saeon.nrf....,2024,2024,https://data.icos-cp.eu/licence_accept?ids=%5B...,SAEON_ZA-Jks_FLUXNET_2024-2024_v1.3_r1.zip,"Joubert, W., Maswanganye, S., Selala, S., Smar...",pJ8V89PtIUTciotXGRT7PLki,v1.3,SAEON
3,ICOS,ZA-BfS,Benfontien Savanna,-28.890600,24.86112,OSH,EFTEON SAEON;European Fluxes Database,Abri de Buys;Helga Knoetz;Siphesihle Faltein;A...,Technician;PI;Technician;Technician;PI;Technician,aj.debuys@saeon.nrf.ac.za;h.vancoller@saeon.nr...,2020,2024,https://data.icos-cp.eu/licence_accept?ids=%5B...,SAEON_ZA-BfS_FLUXNET_2020-2024_v1.3_r1.zip,(2025). Fluxnet Archive Product from Benfontie...,7D87eYcefMtCZ131SjGfAEvu,v1.3,SAEON
4,ICOS,ZA-BfK,Benfontien Karoo,-28.856483,24.83985,OSH,EFTEON SAEON;European Fluxes Database,Abri de Buys;Siphesihle Faltein;Helga Knoetz;A...,Technician;Technician;PI;Technician;Technician;PI,aj.debuys@saeon.nrf.ac.za;s.faltein@saeon.nrf....,2020,2024,https://data.icos-cp.eu/licence_accept?ids=%5B...,SAEON_ZA-BfK_FLUXNET_2020-2024_v1.3_r1.zip,(2025). Fluxnet Archive Product from Benfontie...,L3b447DFrFAeO56M1P7eueuq,v1.3,SAEON


In [7]:
# Filter the data using European country codes and for the time range 2001-2020

european_country_codes = {
    "AT","BE","BG","CH","CZ","DE","DK","EE","ES","FI","FR","GB","GR",
    "HR","HU","IE","IS","IT","LT","LU","LV","NL","NO","PL","PT","RO",
    "SE","SI","SK","BE_g" 
}

catalog["country_code"] = catalog["site_id"].str.split("-").str[0]

is_european = catalog["country_code"].isin(european_country_codes)
overlaps_period = (catalog["first_year"] <= 2020) & (catalog["last_year"] >= 2001)

eu_catalog = catalog[is_european & overlaps_period].copy()

print(f"European sites with data overlapping 2001-2020: {len(eu_catalog)}")
eu_catalog[["site_id", "site_name", "first_year", "last_year"]]

European sites with data overlapping 2001-2020: 143


,site_id,site_name,first_year,last_year
81,SE-Svb,Svartberget,2014,2024
83,SE-Nor,Norunda,2018,2024
84,SE-Htm,Hyltemossa,2018,2024
85,SE-Deg,Degero,2019,2024
87,NL-Loo,Loobos,1997,2024
...,...,...,...,...
304,BE-Lcr,Lochristi,2019,2022
305,AT-Zoe,Zoebelboden,2014,2024
306,AT-PsM,Puergschachen Moor,2015,2019
307,AT-Nsd,Neusiedl,2018,2022


In [8]:
# Download the filtered site level data

site_ids = eu_catalog["site_id"].tolist()

downloaded_files = await download(
    site_ids=site_ids,
    snapshot_file=csv_filename,
    output_dir=download_dir,
)

print(f"Downloaded {len(downloaded_files)} files to {download_dir}")

Downloaded 131 files to /home/mahajan/paper_net_rad/FLUXNET/raw_zips
